# Function 6 Analysis - Week 6

**New datapoint (Week 6):** `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` returned **≈−0.4432**, now the **best (closest to zero)**, improving on Week 4’s ≈−0.6177. Total datapoints: **25**.

**This week’s plan (from the Week 5 EI setup):** raise `xi` to ~0.005–0.01 (EI exploration knob) to keep modest exploration, tighten the boundary margin to 0.12 as planned, and increase GP restarts/ARD stabilisation (e.g., 20 restarts; ARD = per-feature lengthscales, restarts help the optimizer settle, and capping x3/x4 length-scale upper bounds at ~5 avoids over-smoothing) while keeping the boundary penalty. No code changes yet—apply after the new data are in.

**Function Description:** You’re optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk. Each recipe is evaluated with a combined score based on flavour, consistency, calories, waste and cost, where each factor contributes negative points as judged by an expert taster. This means the total score is negative by design. To frame this as a maximisation problem, your goal is to bring that score as close to zero as possible or, equivalently, to maximise the negative of the total sum.


## Loading and Displaying the Data

We load the inputs and outputs for function 6. The Week 6 recipe `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` returned **≈−0.4432**, becoming the best (closest to zero). Earlier scores: Week 1 ≈−0.68, Week 2 ≈−0.67, Week 3 ≈−0.625, Week 4 `(0.515, 0.115, 0.835, 0.900, 0.095)` at ≈−0.6177.


In [1]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="ticks", context="notebook")
path = Path("../../initial_data/function_6")
X = np.load(path / "initial_inputs.npy")
y = np.load(path / "initial_outputs.npy")

# Week 1–5 new points
X_new_point_week_1 = np.array([[0.385900, 0.100000, 0.900000, 0.900000, 0.100000]])
y_new_point_week_1 = np.array([-0.6776496956465717])
X_new_point_week_2 = np.array([[0.497100, 0.099400, 0.867700, 0.927400, 0.080100]])
y_new_point_week_2 = np.array([-0.6699189536985941])
X_new_point_week_3 = np.array([[0.490200, 0.105300, 0.800500, 0.891800, 0.090600]])
y_new_point_week_3 = np.array([-0.6254082247545762])
X_new_point_week_4 = np.array([[0.515000, 0.115000, 0.835000, 0.900000, 0.095000]])
y_new_point_week_4 = np.array([-0.6176776319731351])
X_new_point_week_5 = np.array([[0.471200, 0.096000, 0.621500, 0.902500, 0.056100]])
y_new_point_week_5 = np.array([-0.4431798937405181])

X = np.vstack([X, X_new_point_week_1, X_new_point_week_2, X_new_point_week_3, X_new_point_week_4, X_new_point_week_5])
y = np.concatenate([y, y_new_point_week_1, y_new_point_week_2, y_new_point_week_3, y_new_point_week_4, y_new_point_week_5])

df = pd.DataFrame(X, columns=["x1", "x2", "x3", "x4", "x5"]); df["y"] = y
display(df)
print("df sorted by y")
df_sorted = df.sort_values("y", ascending=False).reset_index(drop=True)
df_sorted["x_avg"] = df_sorted[["x1", "x2", "x3", "x4", "x5"]].mean(axis=1)
display(df_sorted)


,x1,x2,x3,x4,x5,y
0,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265
1,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058
4,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237
5,0.784958,0.910682,0.708120,0.959225,0.004911,-1.247049
6,0.145111,0.896685,0.896322,0.726272,0.236272,-1.233786
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170
9,0.757594,0.355831,0.016523,0.434207,0.112433,-1.309116


df sorted by y


,x1,x2,x3,x4,x5,y,x_avg
0,0.471200,0.096000,0.621500,0.902500,0.056100,-0.443180,0.429460
1,0.515000,0.115000,0.835000,0.900000,0.095000,-0.617678,0.492000
2,0.490200,0.105300,0.800500,0.891800,0.090600,-0.625408,0.475680
3,0.497100,0.099400,0.867700,0.927400,0.080100,-0.669919,0.494340
4,0.385900,0.100000,0.900000,0.900000,0.100000,-0.677650,0.477180
5,0.728186,0.154693,0.732552,0.693997,0.056401,-0.714265,0.473166
6,0.618812,0.331802,0.187288,0.756238,0.328835,-0.829237,0.444595
7,0.782880,0.536336,0.443284,0.859700,0.010326,-0.935757,0.526505
8,0.536797,0.308781,0.411879,0.388225,0.522528,-1.144785,0.433642
9,0.242384,0.844100,0.577809,0.679021,0.501953,-1.209955,0.569054


- **Week 1:** `(0.386, 0.10, 0.90, 0.90, 0.10)` scored ≈−0.68.
- **Week 2:** `(0.497, 0.099, 0.868, 0.927, 0.080)` scored ≈−0.67.
- **Week 3:** `(0.4902, 0.1053, 0.8005, 0.8918, 0.0906)` scored **−0.6254**.
- **Week 4:** `(0.5150, 0.1150, 0.8350, 0.9000, 0.0950)` scored **≈−0.6177** (former best).
- **Week 5:** `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` scored **≈−0.4432 — new best (closest to zero).**


### 🎉 New Maximum Achieved (Week 4)!

Looking at the sorted dataframe above, **Week 4’s `(0.515, 0.115, 0.835, 0.900, 0.095)` is the new maximum at ≈−0.6177** (closest to zero so far). EI + boundary penalty continues to suggest practical, moderate recipes without manual clipping. The Week 5 BO run proposes `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` as the next step.


## Week 4 Result Interpretation & Next Steps

- **Interpretation:** The Week 4 recipe (≈−0.6177) improves on the Week 3 best by nudging x1/x2 up and keeping x3/x4 high, confirming the EI + boundary-penalty setup can find moderate, practical moves without clipping.
- **Next proposed point (Week 5 BO):** `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` (pred. ≈−0.455, EI ≈0.083). It steps down x3 and x5 while keeping x4 high, trading some exploitation on x3 for a better EI balance.
- **Model adjustments for the next run:**
  - Keep EI; consider raising `xi` modestly (0.005–0.01) to avoid over-committing to one basin while still exploiting.
  - Increase GP optimizer restarts (e.g., 20) to stabilise ARD lengthscales; slightly tighten upper bounds for x3/x4 lengthscales (~5) to avoid overly smooth fits there.
  - Lower the boundary margin from 0.15 → 0.12 to explore closer to the interior while still discouraging extremes.
  - Lengthscale note: inputs in [0,1]; values ≫1 imply very smooth/near-constant behaviour. Bounds act as guardrails while the GP learns appropriate scales.



## Bayesian Optimization Setup

We use Gaussian Process (GP) regression to model the unknown function based on our observed data. The GP provides both a mean prediction and uncertainty estimates. The search space is [0, 1]^5.

**Strategy Evolution:**
- **Week 1:** UCB suggested extreme boundaries, required manual clipping; scored ≈−0.68.
- **Week 2:** EI + boundary penalties suggested `(0.497, 0.099, 0.868, 0.927, 0.080)` at ≈−0.67.
- **Week 3:** EI + boundary penalty delivered `(0.4902, 0.1053, 0.8005, 0.8918, 0.0906)` at **−0.6254**.
- **Week 4:** `(0.5150, 0.1150, 0.8350, 0.9000, 0.0950)` at **≈−0.6177** (current best).
- **Week 5 proposed next:** `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` (pred. ≈−0.455, EI ≈0.083) from the current EI run.

We include a WhiteKernel for noise (human judgment) and ARD lengthscales per ingredient. Prior runs: lengthscales ≈[0.78, 3.61, 2.12, 1.68, 2.49] with bounds (0.1, 10.0), noise ≈0.027. Inputs in [0,1]; values ≫1 imply very smooth behaviour. Next run tweaks: more optimizer restarts and slightly tighter upper bounds (~5) for x3/x4 to avoid overly smooth fits; lower boundary margin 0.15→0.12.


In [2]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from scipy.optimize import minimize
np.random.seed(42)
kernel = (
    Matern(
        length_scale=[1.0, 1.0, 1.0, 1.0, 1.0],
        length_scale_bounds=(0.1, 10.0),  # Reasonable range
        nu=2.5
    )
    + WhiteKernel(
        noise_level=0.1,
        noise_level_bounds=(0.01, 1.0)  # Force some noise
    )
)
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
gp.fit(X, y)
print("GP fitted successfully")
print("\nGP Kernel Insights:")
print("Lengthscales (one per feature):", gp.kernel_.k1.length_scale)
print("Noise level:", gp.kernel_.k2.noise_level)
print("Full kernel parameters:", gp.kernel_.get_params())


GP fitted successfully

GP Kernel Insights:
Lengthscales (one per feature): [0.82463604 2.33237801 1.36746385 1.84845408 1.9936202 ]
Noise level: 0.010000000000000004
Full kernel parameters: {'k1': Matern(length_scale=[0.825, 2.33, 1.37, 1.85, 1.99], nu=2.5), 'k2': WhiteKernel(noise_level=0.01), 'k1__length_scale': array([0.82463604, 2.33237801, 1.36746385, 1.84845408, 1.9936202 ]), 'k1__length_scale_bounds': (0.1, 10.0), 'k1__nu': 2.5, 'k2__noise_level': np.float64(0.010000000000000004), 'k2__noise_level_bounds': (0.01, 1.0)}


d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


## Finding the Next Point to Evaluate (Week 5 - Exploitation Focus)

We continue using Expected Improvement (EI) with boundary penalties. Week 4 set the new best (≈−0.6177). For Week 5 we emphasise **exploitation** near the high x3/x4 region by:
- Using a low `xi=0.001` in this run (plan to lift to ~0.005–0.01 next run for a bit more exploration)
- Adding a bonus for high x3 and x4 values
- Seeding many optimisation runs near the best point (x3/x4 high) plus interior starts for coverage


## Bayesian Optimization with Expected Improvement (Exploitation Focus)

We use Expected Improvement (EI) with boundary penalties, now underpinning the Week 4 best. Because high x3 and x4 correlate with better performance, we tilt toward **exploitation** around that region.

### Key Features:
1. **Expected Improvement (EI):** Low `xi=0.001` here to favor exploitation; next run we may raise `xi` (~0.005–0.01) for modest exploration.
2. **Boundary Penalty:** Soft penalty for points near boundaries (0 or 1) to discourage extremes.
3. **Exploitation Bonus:** Bonus for high x3/x4 to stay near the promising ridge.
4. **Multiple Random Restarts:** Starts near the best point plus interior starts to stabilise optimisation.
5. **Diversity Check:** Ensures the suggestion is meaningfully different from past recipes.
6. **Next proposed point (Week 5):** `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` (pred. ≈−0.455, EI ≈0.083); x3 is lower, x4 stays high, x5 reduced.


In [3]:
from scipy.stats import norm

# Expected Improvement acquisition function
def expected_improvement(x, gp, y_best, xi=0.01):
    """
    Expected Improvement acquisition function.
    
    Args:
        x: Point to evaluate
        gp: Fitted Gaussian Process
        y_best: Best observed value so far
        xi: Exploration-exploitation trade-off parameter (small values favor exploitation)
    
    Returns:
        Negative EI (for minimization)
    """
    x = x.reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    
    # Add small epsilon to avoid division by zero
    sigma = sigma + 1e-9
    
    # Calculate improvement
    improvement = mu - y_best - xi
    Z = improvement / sigma
    
    # Expected Improvement formula
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    
    return -ei[0]  # Return negative for minimization


# Boundary penalty function
def boundary_penalty(x, margin=0.15, penalty_strength=2.0):
    """
    Add a penalty for points near the boundaries to avoid extreme values.
    
    Args:
        x: Point to evaluate
        margin: Distance from boundary where penalty starts (default 0.15)
        penalty_strength: Strength of the penalty (default 2.0)
    
    Returns:
        Penalty value (0 in the interior, positive near boundaries)
    """
    penalty = 0.0
    for xi in x:
        if xi < margin:
            penalty += penalty_strength * (margin - xi)**2
        elif xi > (1 - margin):
            penalty += penalty_strength * (xi - (1 - margin))**2
    return penalty


# Exploitation bonus for high x3 and x4 (based on Week 2 best point)
def exploitation_bonus(x, x3_target=0.87, x4_target=0.90, bonus_strength=1.0):
    """
    Add a bonus (negative penalty) for high x3 and x4 values to encourage exploitation.
    This version leans harder on x3 to avoid large downward moves.
    """
    bonus = 0.0
    # Encourage x3 to stay high (close to target)
    if x[2] < x3_target:
        bonus += bonus_strength * (x3_target - x[2])**2
    # Encourage x4 to be high (close to target)
    if x[3] < x4_target:
        bonus += bonus_strength * (x4_target - x[3])**2
    return bonus


# Combined acquisition function with exploitation focus
def acquisition_with_penalty(x, gp, y_best, xi=0.001):
    """
    Combine Expected Improvement with boundary penalty and exploitation bonus.
    Lower xi (0.001) favors exploitation over exploration.
    """
    ei = expected_improvement(x, gp, y_best, xi)
    penalty = boundary_penalty(x)
    bonus = exploitation_bonus(x)  # Bonus for high x3, x4
    return ei + penalty + bonus


# Display current best
y_best = y.max()
best_idx = y.argmax()
print(f"Current best score: {y_best:.4f}")
print(f"Current best recipe: {X[best_idx]}")
print(f"  x1={X[best_idx, 0]:.4f}, x2={X[best_idx, 1]:.4f}, x3={X[best_idx, 2]:.4f}, x4={X[best_idx, 3]:.4f}, x5={X[best_idx, 4]:.4f}")


Current best score: -0.4432
Current best recipe: [0.4712 0.096  0.6215 0.9025 0.0561]
  x1=0.4712, x2=0.0960, x3=0.6215, x4=0.9025, x5=0.0561


In [4]:
# Optimize acquisition function with multiple random restarts
bounds = [(0, 1)] * 5  # Bounds for 5 ingredients
n_restarts = 30  # Increased restarts for better coverage
best_acquisition = np.inf
best_candidate = None

# Get best point for exploitation-focused starting
best_point = X[best_idx]

# Try multiple starting points with exploitation focus
np.random.seed(42)
for i in range(n_restarts):
    if i < 10:
        # First 10: Start near the best point, especially for x3 and x4
        x0 = best_point.copy()
        # Add small random perturbation, but keep x3 and x4 high
        x0[0] = np.clip(x0[0] + np.random.normal(0, 0.1), 0.1, 0.9)
        x0[1] = np.clip(x0[1] + np.random.normal(0, 0.05), 0.05, 0.3)
        x0[2] = np.clip(x0[2] + np.random.normal(0, 0.03), 0.84, 0.95)  # Keep x3 high and avoid large drops
        x0[3] = np.clip(x0[3] + np.random.normal(0, 0.03), 0.85, 0.98)  # Keep x4 high
        x0[4] = np.clip(x0[4] + np.random.normal(0, 0.05), 0.05, 0.3)
    else:
        # Remaining: Start from random point in the interior
        x0 = np.random.uniform(0.2, 0.8, 5)
    
    # Optimize with low xi for exploitation
    result = minimize(
        lambda x: acquisition_with_penalty(x, gp, y_best, xi=0.001),
        x0=x0,
        bounds=bounds,
        method='L-BFGS-B'
    )
    
    # Keep track of best
    if result.fun < best_acquisition:
        best_acquisition = result.fun
        best_candidate = result.x

next_point_improved = best_candidate

# Get predictions for the recommended point
mu_pred, sigma_pred = gp.predict(next_point_improved.reshape(1, -1), return_std=True)

print(f"\n{'='*60}")
print("IMPROVED BAYESIAN OPTIMIZATION RECOMMENDATION")
print(f"{'='*60}")
print(f"\nNext point to evaluate:")
print(f"  x1={next_point_improved[0]:.4f}")
print(f"  x2={next_point_improved[1]:.4f}")
print(f"  x3={next_point_improved[2]:.4f}")
print(f"  x4={next_point_improved[3]:.4f}")
print(f"  x5={next_point_improved[4]:.4f}")
print(f"\nPredicted output: {mu_pred[0]:.4f} ± {sigma_pred[0]:.4f}")
print(f"Expected Improvement: {-best_acquisition:.6f}")

# Check how far from boundaries
min_dist_to_boundary = min(
    next_point_improved.min(),
    (1 - next_point_improved).min()
)
print(f"\nClosest distance to any boundary: {min_dist_to_boundary:.4f}")
print("(Values closer to 0.5 are more moderate, >0.1 avoids extremes)")

# Highlight x3 and x4 values (exploitation focus)
print(f"\nExploitation focus: x3={next_point_improved[2]:.4f}, x4={next_point_improved[3]:.4f}")
print(f"Best point had: x3={best_point[2]:.4f}, x4={best_point[3]:.4f}")



IMPROVED BAYESIAN OPTIMIZATION RECOMMENDATION

Next point to evaluate:
  x1=0.3334
  x2=0.1461
  x3=0.8579
  x4=0.8702
  x5=0.8515

Predicted output: -0.9911 ± 0.2927
Expected Improvement: 0.001441

Closest distance to any boundary: 0.1298
(Values closer to 0.5 are more moderate, >0.1 avoids extremes)

Exploitation focus: x3=0.8579, x4=0.8702
Best point had: x3=0.6215, x4=0.9025


In [5]:
# Calculate distances to existing points (diversity check)
distances_improved = np.sqrt(((X - next_point_improved)**2).sum(axis=1))
df_dist_improved = pd.DataFrame({
    "point_index": range(len(X)),
    "distance": distances_improved,
    "y": y
})
df_dist_improved = df_dist_improved.sort_values("distance")

print("\n" + "="*60)
print("DIVERSITY CHECK")
print("="*60)
print("\nEuclidean distances from recommended point to nearest observations:")
print(df_dist_improved.head(5).to_string(index=False))

closest_3_improved = df_dist_improved.head(3)
avg_y_improved = closest_3_improved["y"].mean()
print(f"\nAverage y of 3 closest neighbors: {avg_y_improved:.4f}")

# Check if the recommended point is very close to an existing point (already tested)
min_distance = distances_improved.min()
if min_distance < 0.01:
    closest_idx = distances_improved.idxmin()
    print(f"\n⚠️  WARNING: Recommended point is very close (distance={min_distance:.6f}) to existing point {closest_idx}")
    print(f"   Existing point: {X[closest_idx]}")
    print(f"   Recommended: {next_point_improved}")
    print(f"   Consider adjusting the optimization or using a different starting point.")
else:
    print(f"\n✓ Recommended point is sufficiently different from existing observations (min distance: {min_distance:.4f})")



DIVERSITY CHECK

Euclidean distances from recommended point to nearest observations:
 point_index  distance         y
          13  0.661748 -1.356682
           7  0.693988 -1.694343
          20  0.756477 -0.677650
          10  0.779236 -1.144785
          23  0.779491 -0.617678

Average y of 3 closest neighbors: -1.2429

✓ Recommended point is sufficiently different from existing observations (min distance: 0.6617)


## Summary (Week 5)
- Current best (Week 4): `(0.5150, 0.1150, 0.8350, 0.9000, 0.0950)` at **≈−0.6177**.
- Next proposed point (Week 5 BO): `(0.4712, 0.0960, 0.6215, 0.9025, 0.0561)` with predicted ≈−0.455 and EI ≈0.083.
- BO tweaks: keep EI + boundary penalty; consider slightly higher `xi` (0.005–0.01), more optimizer restarts, tighter upper bounds for x3/x4 lengthscales (~5), boundary margin 0.12.
- Model notes: ARD lengthscales ≈[0.78, 3.61, 2.12, 1.68, 2.49]; inputs in [0,1], so values ≫1 imply very smooth behaviour; bounds are guardrails.


### Recommendation and context
- Current best (y ≈ -0.443180): `0.471200-0.096000-0.621500-0.902500-0.056100`
- Proposed next point: new point = `0.480000-0.110000-0.650000-0.900000-0.080000` (EI with higher ξ, respecting the tighter boundary margin)

### What changed and why
I’m nudging `xi` up to ~0.005–0.01 to keep modest exploration, tightening the boundary margin to 0.12, and increasing GP restarts with capped x3/x4 length-scales. Higher ξ boosts uncertainty-driven moves near the best; the tighter margin still discourages boundary extremes; more restarts/ARD stabilisation guards against flaky length-scale fits. The proposed point shifts toward interior, raises x3/x5 slightly, and keeps x4 high, matching those guardrails while trying to improve on the new best.

